# Scale up the PROVEN backbone — thousands of GAMUS tiles, not the experimental HTC head

The production checkpoint (`checkpoints/height_best/`) was fine-tuned on only **400** of the **3,837** GAMUS train tiles actually available, and it already works (MAE 2.439m -> 1.444m, correlation 0.784 -> 0.912 on GAMUS's own held-out split; cross-domain on DFC2019, 3.063m per-building RMSE, ahead of a competitor's reported 3.464m). This notebook does the highest-confidence thing available to push that further: more data on the SAME architecture that is already licensed and shipping, not a bet on the still-unproven HTC-AdaBins head (see kaggle_htc_train.ipynb for that separate, riskier experiment).

No DFC2019 upload needed for this one -- pure GAMUS, which downloads directly over HTTP inside the notebook.

**Before running:**
1. Settings -> Accelerator -> **GPU T4 x2** (needs phone verification once, if not done already). Internet: On.
2. Kaggle Secret `GITHUB_TOKEN` attached (same one from kaggle_htc_train.ipynb works here too).

That's it -- no dataset upload step this time.

In [ ]:
# Cell 1: clone the private repo
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")

!rm -rf /kaggle/working/depthwizard
!git clone --depth 1 https://{token}@github.com/jawaharananth/depthwizard.git /kaggle/working/depthwizard
%cd /kaggle/working/depthwizard

In [ ]:
# Cell 2: dependencies
!pip install -q transformers h5py rasterio opencv-python-headless

In [ ]:
# Cell 3: download a real 2,500-tile GAMUS subset (out of 3,837 available) plus a 400-tile
# val set. This alone is 6x the 400 tiles the current production checkpoint was trained on.
# gamus.py's `--limit` spreads the sample across the tile list rather than taking a prefix,
# so this pulls from both PHL and DC, not just whichever city sorts first.
# ~80GB dataset total, but we're only pulling the subset we ask for -- expect this to take
# a while depending on Kaggle's network to HuggingFace; it's the long pole before training
# even starts.
!python gamus.py download --split train --limit 2500
!python gamus.py download --split val --limit 400

In [ ]:
# Cell 4: the real run. Same architecture as the currently-licensed checkpoint (plain
# regression, NOT --htc), just far more data and a real epoch budget.
# batch=8, accum=2 (effective batch 16) sized for a T4's ~15GB VRAM at crop=518 with the
# Base backbone -- drop --batch if this OOMs, raise it on a P100 or better card.
# 2,500 tiles x 4 crops/tile = 10,000 crops/epoch; at a rough 15-20 crops/s on a T4 that's
# roughly 10-15 min/epoch, so 15 epochs is a ~3-4 hour session -- well inside Kaggle's free
# per-session GPU cap. Raise --epochs if the session has budget left; the script only ever
# keeps the best-MAE checkpoint, so a longer run can only help, never regress the result.
!python train_height.py \
    --epochs 15 --batch 8 --accum 2 --crops-per-tile 4 --workers 2 \
    --out /kaggle/working/checkpoints

In [ ]:
# Cell 5: package for download
!zip -r /kaggle/working/metric_checkpoint.zip /kaggle/working/checkpoints
print("Download metric_checkpoint.zip from this notebook's Output tab when the run finishes.")
print("It contains height_best/ (model.safetensors, config.json, preprocessor_config.json,")
print("metric_ok.json ONLY if this run beat the baseline) and training_report.json.")

## After downloading

Compare `training_report.json`'s `best_mae_m` against the current local checkpoint's `checkpoints/height_best/metric_ok.json` (`finetuned_mae_m_no_affine: 1.444`). If `metric_ok.json` is present in the new zip AND its `finetuned_mae_m_no_affine` beats 1.444, copy the whole `height_best/` folder over the local one -- `depth_model.py` picks it up automatically via `DepthBackbone.metric_available()`. If `metric_ok.json` is absent, this run did not beat the *new* run's own baseline (a stricter bar than 400-tile training had, since the baseline is measured fresh each run) -- do not copy it over, same licensing rule as always.